# 21a · Retina revisions, review cycle 1 (post hoc)

**Status: post hoc, requested in review.** This notebook answers points raised in the pre-submission review of 29 October 2026. It reads only frozen v2 prediction files and changes no locked result. Every result is reported whatever it shows. Only aggregate numbers are written.

| Section | Review point | What it does |
|---|---|---|
| R1 | 5.5 | Union of both systems' referrals, with the **better member chosen on the outcome each row reports** (as in skin notebook 20 v4), in EyePACS, DDR and BRSET. |
| R2 | 3.2 | **Full-procedure bootstrap**: each resample of patients (or DDR filename groups) rebuilds every system's ranks, referral set and pair referral set from scratch. Checks the EyePACS primary result, the DDR P1 result and the BRSET B2a result. |
| R3 | 2.4 | **Patient-level** summaries in EyePACS and BRSET (DDR releases no patient identifiers): a patient is referred if any image is referred, and a patient with severe DR is missed if none of their images is referred. Rescues and newly introduced misses are reported separately, with the referral workload in patients. |
| R4 | 2.1 | **Fixed-anchor training-source comparison** in BRSET: each system keeps its own referrals as the anchor and adds either a partner trained on the same source or one trained on the other source, of the same design. Absolute and relative net misses avoided, and a comparison adjusted for partner strength. |
| R7 | 3.1 | Retinal degradation-fingerprint analysis (BALLISTICS full recipe) re-estimated with a **model bootstrap that keeps repeated draws**. |

Sections R5 (patient-disjoint partner choice) and R6 (out-of-sample harm tuning) are in notebook 21b so the two can run in parallel.

**Runtime:** CPU, High-RAM, about 60 to 90 minutes (R2 is the slow part).

In [1]:
# --- setup ---
from google.colab import drive
drive.mount("/content/drive")
import os, re, json, hashlib, itertools, datetime, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import rankdata
BASE = "/content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2"
PREDS, SPLITS, RESULTS, FIGS = (f"{BASE}/{d}" for d in ("03_predictions", "01_splits", "04_results", "05_figures"))
BRSET_LABELS = "/content/drive/MyDrive/stochastic_orthogonality/data/brset/label_brset.csv"
OUTD = f"{RESULTS}/revisions_c1"; os.makedirs(OUTD, exist_ok=True)
OUT_JSON = f"{OUTD}/retina_revisions_c1_a.json"
assert not os.path.exists(OUT_JSON), "this secondary analysis has already been written; rename the old output deliberately to rerun"
def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()
ARCHS = ["densenet121_binary", "densenet121_5class", "efficientnet_b3_binary", "efficientnet_b3_5class", "resnet50_binary", "resnet50_5class",
         "vit_base_binary", "vit_base_5class", "retfound_binary", "retfound_5class", "retfound_adversarial"]
SEEDS = [0, 1, 2]; PAIRS = list(itertools.combinations(ARCHS, 2))
def tag(k): return "" if k == 0 else f"_seed{k}"
MODELS = [(a, k) for a in ARCHS for k in SEEDS]
B, SEED = 1000, 20261028
VOLS = np.round(np.arange(0.01, 0.701, 0.01), 2)

# harm model (severe visual loss events per missed eye, one-year delay)
PDR = {"low": 0.02, "central": 0.0475, "high": 0.075}
P_HR_G3, DIRECT_G3 = 0.15, 0.002
P_PDR_G2 = {"low": 0.02, "central": 0.04, "high": 0.08}
SY = {"low": 10, "central": 15, "high": 20}; QALY_PER_BLIND_YEAR = 0.214
def svl(grade, sc):
    r = PDR[sc]
    return np.select([grade == 4, grade == 3, grade == 2], [r, P_HR_G3 * 0.5 * r + DIRECT_G3, P_PDR_G2[sc] * 0.5 * r], 0.0)
KS = np.round(np.geomspace(10, 10000, 31), 1)          # false referrals one prevented severe visual loss event is worth
LAMBDAS = np.round(np.geomspace(1e-4, 3e-2, 25), 6); BAND = (0.001, 0.004); LAM_ANCHOR = 0.0025
NPJ_GRID = {"miss": [[500, 200, 15], [250, 100, 10], [1000, 400, 20], [100, 50, 10], [50, 25, 5]], "wfp": [1, 5, 20, 50, 100]}
for sc in ("low", "central", "high"):
    print(f"{sc:>7}: per missed eye, severe visual loss events  g4 {svl(np.array(4), sc):.4f}  g3 {svl(np.array(3), sc):.4f}  g2 {svl(np.array(2), sc):.5f}"
          f"  | QALY per event {SY[sc] * QALY_PER_BLIND_YEAR:.2f}")
CORE4 = ["densenet121_5class", "efficientnet_b3_5class", "resnet50_5class", "retfound_5class"]
FULL = f"{BASE}/04_results/ballistics_full"


Mounted at /content/drive
    low: per missed eye, severe visual loss events  g4 0.0200  g3 0.0035  g2 0.00020  | QALY per event 2.14
central: per missed eye, severe visual loss events  g4 0.0475  g3 0.0056  g2 0.00095  | QALY per event 3.21
   high: per missed eye, severe visual loss events  g4 0.0750  g3 0.0076  g2 0.00300  | QALY per event 4.28


## 1. Load (frozen predictions only)

In [2]:
def pick(cols, pats, what):
    m = [c for c in cols if any(re.fullmatch(p, c, flags=re.I) for p in pats)]
    assert len(m) == 1, f"cannot identify the {what} column; columns are {list(cols)}"
    return m[0]
F_OVERRIDE = {}   # e.g. {"ddr": 0.56} if the printed f differs from the locked value
def norm(x): return pd.Series(x, dtype=str).str.replace(r"\.(jpe?g|png|tiff?)$", "", regex=True, flags=re.I).to_numpy()
HASHES = {}
def load_preds(fmt, key="image_id"):
    P = {}; ref = None
    for a, k in MODELS:
        fn = f"{PREDS}/{fmt.format(arch=a, tag=tag(k))}"; HASHES[os.path.basename(fn)] = sha256_file(fn)
        d = pd.read_csv(fn, dtype={key: str})
        if ref is None: ref = d
        else: assert (d[key].values == ref[key].values).all(), f"row order differs in {os.path.basename(fn)}"
        P[(a, k)] = d["predicted_probability"].to_numpy(np.float64)
    return ref, P
POP = {}
# EyePACS fresh set (labels and patients travel with the prediction files)
ref, P = load_preds("eyepacs_predictions_fresh_{arch}{tag}_v2.csv")
POP["eyepacs"] = {"grade": ref.true_grade.to_numpy(int), "group": ref.patient_id.astype(str).to_numpy(), "P": P, "f": 0.25}
# DDR (labels and filename groups from the intraset split written before inference)
ref, P = load_preds("ddr_predictions_{arch}{tag}_v2.csv")
sp = pd.read_csv(f"{SPLITS}/ddr_split_intraset_v2.csv", dtype={"image_id": str})
gcol, grpcol = pick(sp.columns, ["grade", "true_grade", "dr_grade", "level"], "DDR grade"), pick(sp.columns, ["group", "filename_group", "group_id"], "DDR group")
sp["_k"] = norm(sp["image_id"]); sp = sp.set_index("_k").loc[norm(ref.image_id)]
half = [c for c in sp.columns if re.fullmatch("half|split|subset", c, flags=re.I)]
g = sp[gcol].to_numpy(int); keep = g <= 4
if half:
    isA = sp[half[0]].astype(str).str.strip().str.upper().isin(["A", "HALF_A", "HALFA", "0"]).to_numpy()
    print("DDR half column:", half[0], "| values:", sorted(sp[half[0]].astype(str).unique())[:6], "| treated as half A:", int((isA & keep).sum()), "eyes")
else:
    isA = np.ones(len(g), bool); print("WARNING: no DDR half column found; f uses the whole DDR prevalence")
prevA = float((g[keep & isA] >= 2).mean())
POP["ddr"] = {"grade": g[keep], "group": sp[grpcol].astype(str).to_numpy()[keep], "P": {m: v[keep] for m, v in P.items()}, "f": round(1.31 * prevA, 4)}
# BRSET (labels read from the BRSET label file; only aggregates leave this runtime)
ref, P = load_preds("brset_predictions_eyepacs_{arch}{tag}_v2.csv")
lb = pd.read_csv(BRSET_LABELS, dtype=str)
icol, pcol, idcol = pick(lb.columns, [r".*icdr.*"], "BRSET ICDR grade"), pick(lb.columns, ["patient_id", "patient"], "BRSET patient"), pick(lb.columns, ["image_id", "image"], "BRSET image id")
lb["_k"] = norm(lb[idcol]); lb = lb.set_index("_k").loc[norm(ref.image_id)]
g = pd.to_numeric(lb[icol], errors="coerce").to_numpy(); keep = np.isin(g, [0, 1, 2, 3, 4])
POP["brset"] = {"grade": g[keep].astype(int), "group": lb[pcol].astype(str).to_numpy()[keep], "P": {m: v[keep] for m, v in P.items()}}
POP["brset"]["f"] = round(1.31 * float((POP["brset"]["grade"] >= 2).mean()), 4)
for name, v in F_OVERRIDE.items(): POP[name]["f"] = v
for name, d in POP.items():
    gr = d["grade"]
    print(f"{name:>8}: {len(gr):,} eyes | {len(set(d['group'])):,} groups | referable {np.mean(gr >= 2):.3f} | grade 3 {int((gr == 3).sum())} | grade 4 {int((gr == 4).sum())} | f = {d['f']}")
print("expected f from the manuscript: DDR about 0.56 (from half A), BRSET about 0.075")
# DDR-trained models on BRSET (for the fixed-anchor training-source comparison)
ids_b = norm(pd.read_csv(f"{PREDS}/brset_predictions_eyepacs_{ARCHS[0]}_v2.csv", dtype={"image_id": str}).image_id)
keepb = np.isin(pd.to_numeric(lb[icol], errors="coerce").to_numpy(), [0, 1, 2, 3, 4])
PB = {}
for src in ("eyepacs", "ddr"):
    for a in CORE4:
        for k in SEEDS:
            fn = f"{PREDS}/brset_predictions_{src}_{a}{tag(k)}_v2.csv"; HASHES[os.path.basename(fn)] = sha256_file(fn)
            d = pd.read_csv(fn, dtype={"image_id": str}); assert (norm(d.image_id) == ids_b).all(), fn
            PB[(src, a, k)] = d["predicted_probability"].to_numpy(np.float64)[keepb]
POP["brset"]["PB"] = PB
print("BRSET core models (both training sources):", len(PB))


DDR half column: half | values: ['A', 'B'] | treated as half A: 6205 eyes
 eyepacs: 15,794 eyes | 7,897 groups | referable 0.195 | grade 3 384 | grade 4 328 | f = 0.25
     ddr: 12,422 eyes | 332 groups | referable 0.453 | grade 3 236 | grade 4 913 | f = 0.555
   brset: 16,266 eyes | 8,524 groups | referable 0.057 | grade 3 78 | grade 4 396 | f = 0.0745
expected f from the manuscript: DDR about 0.56 (from half A), BRSET about 0.075
BRSET core models (both training sources): 24


In [3]:

def refer(s, f):
    r = np.zeros(len(s), bool); r[np.argsort(-s, kind="stable")[:int(round(f * len(s)))]] = True; return r
def rk(s): return rankdata(s) / len(s)
def pairscore(ra, rb): return np.maximum(ra, rb) + 1e-9 * np.minimum(ra, rb)
def boot_w(groups, B, seed):
    codes = pd.factorize(groups)[0]; G = codes.max() + 1; rng = np.random.default_rng(seed)
    W = np.empty((B + 1, len(codes)), np.float32); W[0] = 1.0
    for b in range(1, B + 1): W[b] = np.bincount(rng.integers(0, G, G), minlength=G)[codes]
    return W
def ci(x):
    x = np.asarray(x, float); f = x[1:][np.isfinite(x[1:])]
    return [float(x[0]), float(np.percentile(f, 2.5)) if len(f) else None, float(np.percentile(f, 97.5)) if len(f) else None]
def ratio_ci(num, den):
    r = np.where(den > 0, num / np.where(den > 0, den, 1), np.inf)
    return float(r[0]), float(np.percentile(r[1:], 2.5)), float(np.percentile(r[1:], 97.5)), float(np.mean(~np.isfinite(r[1:])))
def fmt(t, d=2): return f"{t[0]:.{d}f} ({t[1]:.{d}f} to {t[2]:.{d}f})"
REC = {"status": "secondary and exploratory analyses requested in review cycle 1 (Astra, 29 Oct 2026); post hoc; no lock",
       "written_utc": None, "bootstrap_resamples": B}
print("helpers ready")


helpers ready


## R1. Union with the better member chosen on the outcome reported
For each pair, each system refers its own top f, and the union refers anything either flags. Against the **better member**, the member is chosen in each resample on the outcome in that row: fewer missed severe eyes for the severe-count row; fewer modelled severe-visual-loss events for the literature-harm row; fewer events with every missed severe eye counted as lost for that row. Against the **average member**, the two members are averaged. Costs are extra false referrals (grade 0 or 1) per unit averted, with bootstrap 95% intervals computed in every resample from the pair-averaged numerator and denominator. The severe-count-chosen version (as in notebook 18b) is printed alongside for comparison.

In [4]:
HR = {"severe_count": None, "lit_central": "central", "allblind": "allblind"}
def contrib(gr):
    c = {"severe_count": (gr >= 3).astype(float), "fp": (gr <= 1).astype(float), "lit_central": svl(gr, "central")}
    c["allblind"] = np.select([gr >= 3, gr == 2], [1.0, svl(np.array(2), "high")], 0.0); return c
R1 = []
for name, d in POP.items():
    t0 = time.time(); gr = d["grade"]; f = d["f"]; W = boot_w(d["group"], B, SEED); Wt = W.sum(1, dtype=np.float64); cb = contrib(gr)
    own = {m: refer(p, f) for m, p in d["P"].items()}
    def miss(ref, h): return W @ ((~ref) * cb[h]).astype(np.float32)
    def fps(ref): return W @ (ref * cb["fp"]).astype(np.float32)
    acc = {}
    for a, bm in PAIRS:
        for i in SEEDS:
            for j in SEEDS:
                ma, mb = (a, i), (bm, j); u = own[ma] | own[mb]; fa, fb, fu = fps(own[ma]), fps(own[mb]), fps(u)
                sev_a, sev_b = miss(own[ma], "severe_count"), miss(own[mb], "severe_count")
                for h in HR:
                    ha, hb, hu = miss(own[ma], h), miss(own[mb], h), miss(u, h)
                    for chooser, (ca, cbb) in (("outcome", (ha, hb)), ("severe_count", (sev_a, sev_b))):
                        bt = ca <= cbb
                        acc.setdefault(("better", chooser, h), [[], []])
                        acc[("better", chooser, h)][0].append(fu - np.where(bt, fa, fb))
                        acc[("better", chooser, h)][1].append(np.where(bt, ha, hb) - hu)
                    acc.setdefault(("average", "none", h), [[], []])
                    acc[("average", "none", h)][0].append(fu - 0.5 * (fa + fb)); acc[("average", "none", h)][1].append(0.5 * (ha + hb) - hu)
    for (cmp_, chooser, h), (num, den) in acc.items():
        num = np.mean(num, 0) / Wt * 1e4; den = np.mean(den, 0) / Wt * 1e4
        c = ratio_ci(num, den)
        R1.append({"population": name, "comparator": cmp_, "better_chosen_on": chooser, "outcome": h, "extra_false_referrals_per_10k": ci(num), "averted_per_10k": ci(den),
                   "false_referrals_per_unit_averted": c[:3], "share_resamples_infinite": c[3]})
    print(f"{name}: done ({time.time() - t0:.0f}s)")
R1df = pd.DataFrame([{**{k: v for k, v in r.items() if not isinstance(v, list)}, "cost": fmt(r["false_referrals_per_unit_averted"], 0), "averted": fmt(r["averted_per_10k"], 3)} for r in R1])
pd.set_option("display.width", 220, "display.max_rows", 200); display(R1df)
REC["R1_union_outcome_chosen_better"] = R1

eyepacs: done (46s)
ddr: done (35s)
brset: done (46s)


,population,comparator,better_chosen_on,outcome,false_referrals_per_unit_averted,share_resamples_infinite,cost,averted
0,eyepacs,better,outcome,severe_count,"(59.370468214453965, 48.762488928278195, 80.75...",0.0,59 (49 to 81),10.092 (7.371 to 12.139)
1,eyepacs,better,severe_count,severe_count,"(59.370468214453965, 48.762488928278195, 80.75...",0.0,59 (49 to 81),10.092 (7.371 to 12.139)
2,eyepacs,average,none,severe_count,"(42.17732783650912, 34.62430017119436, 54.2625...",0.0,42 (35 to 54),13.878 (10.963 to 16.796)
3,eyepacs,better,outcome,lit_central,"(1563.0120706022988, 1334.5692658430507, 1949....",0.0,1563 (1335 to 1950),0.383 (0.309 to 0.446)
4,eyepacs,better,severe_count,lit_central,"(1551.7086307109826, 1309.558673757012, 1903.0...",0.0,1552 (1310 to 1903),0.386 (0.316 to 0.455)
5,eyepacs,average,none,lit_central,"(1146.0077598375208, 954.7412298700766, 1366.5...",0.0,1146 (955 to 1367),0.511 (0.430 to 0.606)
6,eyepacs,better,outcome,allblind,"(56.881629868513286, 47.11682565052225, 75.987...",0.0,57 (47 to 76),10.536 (7.834 to 12.603)
7,eyepacs,better,severe_count,allblind,"(56.86599406158158, 47.08367472750128, 75.9698...",0.0,57 (47 to 76),10.537 (7.835 to 12.604)
8,eyepacs,average,none,allblind,"(40.78270256662331, 33.68609483814433, 51.9330...",0.0,41 (34 to 52),14.353 (11.455 to 17.278)
9,ddr,better,outcome,severe_count,"(43.000876691821254, 25.943423187886797, 85.74...",0.0,43 (26 to 86),14.843 (7.853 to 22.499)


## R2. Full-procedure bootstrap
Each resample draws patients (EyePACS, BRSET) or filename groups (DDR) with replacement and keeps every image of a drawn cluster as many times as it was drawn. Within the resample, every system's score is re-ranked, its top f is re-referred, and each pair's maximum-rank score is re-referred at the same f. The statistic is then recomputed exactly as in its plan. This removes the conditioning on full-sample ranks used by the other intervals.

- **EyePACS (lock v3 primary):** mean over 55 pairs × 9 seed combinations of the better member's missed severe eyes (grade 3 or 4) minus the pair's, at f = 25%. Locked estimate 4.741 (2.31 to 6.15).
- **DDR (lock v5.1 P1):** pooled relative reduction in missed PDR against the average member, 110 ordered pairs × 9, at f = 1.31 × half-A prevalence. Locked estimate 27.4%.
- **BRSET (plan v6 B2a):** pooled relative reduction in missed severe DR against the average member, 110 ordered pairs × 9, at f = 1.31 × prevalence. Locked estimate 24.7%.

In [5]:
NB_FULL = 1000
def full_stat(gr, P, f, kind):
    R = {m: refer(p, f) for m, p in P.items()}; K = {m: rk(p) for m, p in P.items()}
    tgt = (gr >= 3) if kind in ("eyepacs", "brset") else (gr == 4)
    if kind == "eyepacs":
        v = []
        for a, bm in PAIRS:
            for i in SEEDS:
                for j in SEEDS:
                    ma, mb = (a, i), (bm, j); e = refer(pairscore(K[ma], K[mb]), f)
                    v.append(min((tgt & ~R[ma]).sum(), (tgt & ~R[mb]).sum()) - (tgt & ~e).sum())
        return float(np.mean(v))
    anc = ens = 0
    for a, bm in PAIRS:
        for i in SEEDS:
            for j in SEEDS:
                ma, mb = (a, i), (bm, j); e = (tgt & ~refer(pairscore(K[ma], K[mb]), f)).sum()
                anc += (tgt & ~R[ma]).sum() + (tgt & ~R[mb]).sum(); ens += 2 * e
    return float((anc - ens) / anc) if anc else np.nan
R2 = {}
for name in ("eyepacs", "ddr", "brset"):
    d = POP[name]; t0 = time.time(); codes = pd.factorize(d["group"])[0]; G = codes.max() + 1
    members = [np.flatnonzero(codes == g) for g in range(G)]; rng = np.random.default_rng(SEED + 7)
    obs = full_stat(d["grade"], d["P"], d["f"], name); bs = []
    for b in range(NB_FULL):
        idx = np.concatenate([members[g] for g in rng.integers(0, G, G)])
        bs.append(full_stat(d["grade"][idx], {m: p[idx] for m, p in d["P"].items()}, d["f"], name))
        if b in (9, 99, 499): print(f"  {name}: {b + 1} resamples ({time.time() - t0:.0f}s)")
    R2[name] = {"statistic": "net severe misses avoided per pair vs better member" if name == "eyepacs" else ("relative reduction, missed PDR, vs average member" if name == "ddr" else "relative reduction, missed severe DR, vs average member"),
                "estimate": obs, "full_procedure_ci95": [float(np.nanpercentile(bs, 2.5)), float(np.nanpercentile(bs, 97.5))], "resamples": NB_FULL}
    print(name, R2[name])
REC["R2_full_procedure_bootstrap"] = R2

  eyepacs: 10 resamples (13s)
  eyepacs: 100 resamples (120s)
  eyepacs: 500 resamples (586s)
eyepacs {'statistic': 'net severe misses avoided per pair vs better member', 'estimate': 4.739393939393939, 'full_procedure_ci95': [2.2275757575757575, 6.275252525252525], 'resamples': 1000}
  ddr: 10 resamples (10s)
  ddr: 100 resamples (91s)
  ddr: 500 resamples (437s)
ddr {'statistic': 'relative reduction, missed PDR, vs average member', 'estimate': 0.27344481605351173, 'full_procedure_ci95': [0.17398423046792613, 0.38994062149711256], 'resamples': 1000}
  brset: 10 resamples (13s)
  brset: 100 resamples (119s)
  brset: 500 resamples (589s)
brset {'statistic': 'relative reduction, missed severe DR, vs average member', 'estimate': 0.24685897435897436, 'full_procedure_ci95': [0.19444558575939216, 0.3057430667021109], 'resamples': 1000}


## R3. Patient-level summaries
In EyePACS and BRSET (DDR has no patient identifiers; its filename groups hold dozens of images each, so a group-level summary would be almost uninformative), averaged over all 55 pairs × 9 seed combinations and over both members as the first system:
- images and patients;
- patients referred (any image referred), for the single system and for the pair, at the same image-level referral share;
- patients with severe DR (grade 3 or 4 in any image) missed, and separately **rescued** (missed by the first system, caught by the pair) and **newly missed** (caught by the first system, missed by the pair);
- the same against the better member, chosen on patient-level severe misses.

Intervals resample patients.

In [6]:
R3 = {}
for name in ("eyepacs", "brset"):                  # DDR filename groups hold many images each, so a group-level summary is not meaningful
    d = POP[name]
    t0 = time.time(); gr = d["grade"]; f = d["f"]; pid = pd.factorize(d["group"])[0]; npat = pid.max() + 1
    W = boot_w(np.arange(npat), B, SEED + 11); Wt = W.sum(1, dtype=np.float64)
    def to_pat(v): return np.bincount(pid, weights=v.astype(float), minlength=npat) > 0
    sevp = to_pat(gr >= 3); refp = to_pat(gr >= 2)
    own = {m: refer(p, f) for m, p in d["P"].items()}; K = {m: rk(p) for m, p in d["P"].items()}
    ownp = {m: to_pat(r) for m, r in own.items()}
    S = {k: np.zeros((B + 1,)) for k in ("first_missed", "pair_missed", "rescued", "newly_missed", "first_referred", "pair_referred", "better_missed", "first_false_pat", "pair_false_pat")}
    cnt = 0
    for a, bm in PAIRS:
        for i in SEEDS:
            for j in SEEDS:
                ma, mb = (a, i), (bm, j); ep = to_pat(refer(pairscore(K[ma], K[mb]), f)); pm = sevp & ~ep
                am, bmiss = sevp & ~ownp[ma], sevp & ~ownp[mb]
                bt = (W @ am.astype(np.float32)) <= (W @ bmiss.astype(np.float32))
                S["better_missed"] += np.where(bt, W @ am.astype(np.float32), W @ bmiss.astype(np.float32))
                for x in (ma, mb):
                    xm = sevp & ~ownp[x]
                    S["first_missed"] += W @ xm.astype(np.float32); S["pair_missed"] += W @ pm.astype(np.float32)
                    S["rescued"] += W @ (xm & ep).astype(np.float32); S["newly_missed"] += W @ (~xm & sevp & pm).astype(np.float32)
                    S["first_referred"] += W @ ownp[x].astype(np.float32); S["pair_referred"] += W @ ep.astype(np.float32)
                    S["first_false_pat"] += W @ (ownp[x] & ~refp).astype(np.float32); S["pair_false_pat"] += W @ (ep & ~refp).astype(np.float32)
                    cnt += 1
    for k in S: S[k] = S[k] / (cnt if k != "better_missed" else cnt / 2)
    out = {"images": int(len(gr)), "patients_or_groups": int(npat), "patients_with_severe": int(sevp.sum()), "unit": "patient"}
    for k, v in S.items(): out[f"{k}_per_10k_patients"] = ci(v / Wt * 1e4)
    out["net_vs_average_member_per_10k"] = ci((S["first_missed"] - S["pair_missed"]) / Wt * 1e4)
    out["net_vs_better_member_per_10k"] = ci((S["better_missed"] - S["pair_missed"]) / Wt * 1e4)
    out["relative_reduction_vs_average_member"] = ci((S["first_missed"] - S["pair_missed"]) / S["first_missed"])
    out["patient_referral_rate_first_vs_pair"] = [float(S["first_referred"][0] / Wt[0]), float(S["pair_referred"][0] / Wt[0])]
    R3[name] = out; print(name, f"({time.time() - t0:.0f}s)"); print({k: (fmt(v, 2) if isinstance(v, list) and len(v) == 3 and None not in v else v) for k, v in out.items()})
REC["R3_patient_level"] = R3

eyepacs (32s)
{'images': 15794, 'patients_or_groups': 7897, 'patients_with_severe': 426, 'unit': 'patient', 'first_missed_per_10k_patients': '15.58 (9.78 to 21.87)', 'pair_missed_per_10k_patients': '12.31 (7.28 to 18.31)', 'rescued_per_10k_patients': '6.52 (4.56 to 8.71)', 'newly_missed_per_10k_patients': '3.25 (2.04 to 4.78)', 'first_referred_per_10k_patients': '3400.45 (3314.38 to 3484.99)', 'pair_referred_per_10k_patients': '3362.87 (3275.42 to 3448.59)', 'better_missed_per_10k_patients': '13.47 (7.52 to 18.50)', 'first_false_pat_per_10k_patients': '1639.63 (1584.59 to 1694.67)', 'pair_false_pat_per_10k_patients': '1567.07 (1509.24 to 1625.07)', 'net_vs_average_member_per_10k': '3.27 (1.68 to 4.85)', 'net_vs_better_member_per_10k': '1.16 (-1.05 to 1.83)', 'relative_reduction_vs_average_member': '0.21 (0.11 to 0.33)', 'patient_referral_rate_first_vs_pair': [0.3400447427293065, 0.33628701859675647]}
brset (33s)
{'images': 16266, 'patients_or_groups': 8524, 'patients_with_severe': 299,

## R4. Fixed-anchor training-source comparison (BRSET)
Roster: the four 5-class designs trained on each source (EyePACS, DDR), 3 seeds each, 24 systems. For each anchor system and each other design, the anchor is paired with that design trained on the **same** source or on the **other** source (all 3 seeds of each). The statistic is the anchor's missed cases minus the pair's, at the same referral share f: absolute (per 10,000 images) and relative to the anchor's misses.

Partner strength differs by source (DDR-trained systems were weaker on BRSET), so three views are given:
1. **Unadjusted** mean, different-source minus same-source partner.
2. **Strength-matched**: for each anchor and partner design, the different-source and same-source seeds are paired by closest standalone miss count on that outcome; only pairs within 10% of each other are kept.
3. **Adjusted**: regression of net misses avoided on a different-source indicator and the partner's standalone misses, with anchor fixed effects.

Shown overall and separately for EyePACS-trained and DDR-trained anchors. Outcomes: referable DR (the plan v6 B1 outcome), severe DR (grade 3 or 4) and PDR. Intervals resample patients.

In [7]:
d = POP["brset"]; gr = d["grade"]; f = d["f"]; PB = d["PB"]; W = boot_w(d["group"], B, SEED + 13); Wt = W.sum(1, dtype=np.float64)
OUTC = {"referable": gr >= 2, "severe": gr >= 3, "pdr": gr == 4}
RB = {m: refer(p, f) for m, p in PB.items()}; KB = {m: rk(p) for m, p in PB.items()}
R4 = {}
for oname, y in OUTC.items():
    units = []; vecs = []
    for an in PB:
        s, a, k = an
        for a2 in CORE4:
            if a2 == a: continue
            for s2 in ("eyepacs", "ddr"):
                for k2 in SEEDS:
                    pt = (s2, a2, k2); e = refer(pairscore(KB[an], KB[pt]), f)
                    vecs.append(((y & ~RB[an]).astype(np.int8) - (y & ~e).astype(np.int8)))
                    units.append({"anchor": an, "anchor_source": s, "partner": pt, "partner_design": a2, "diff_source": int(s2 != s),
                                  "anchor_miss": int((y & ~RB[an]).sum()), "partner_miss": int((y & ~RB[pt]).sum())})
    U = pd.DataFrame(units); V = np.stack(vecs).astype(np.float32); NET = (W @ V.T)            # (B+1) x units
    AM = np.array([W @ (y & ~RB[u]).astype(np.float32) for u in U.anchor]).T
    def summ(mask):
        dd_, ss_ = mask & (U.diff_source == 1).to_numpy(), mask & (U.diff_source == 0).to_numpy()
        net_d = NET[:, dd_].mean(1) / Wt * 1e4; net_s = NET[:, ss_].mean(1) / Wt * 1e4
        rel_d = NET[:, dd_].sum(1) / AM[:, dd_].sum(1); rel_s = NET[:, ss_].sum(1) / AM[:, ss_].sum(1)
        return {"diff_source_net_per_10k": ci(net_d), "same_source_net_per_10k": ci(net_s), "diff_minus_same_per_10k": ci(net_d - net_s),
                "diff_source_relative": ci(rel_d), "same_source_relative": ci(rel_s), "diff_minus_same_relative": ci(rel_d - rel_s),
                "partner_miss_mean_diff_vs_same": [float(U.partner_miss[dd_].mean()), float(U.partner_miss[ss_].mean())], "units": int(mask.sum())}
    res = {}
    for lab, m in (("all_anchors", np.ones(len(U), bool)), ("eyepacs_anchors", (U.anchor_source == "eyepacs").to_numpy()), ("ddr_anchors", (U.anchor_source == "ddr").to_numpy())):
        res[lab] = summ(m)
        # strength-matched pairs of units
        idx_d, idx_s = [], []
        for (an, a2), g in U[m].groupby(["anchor", "partner_design"]):
            gd, gs = g[g.diff_source == 1], g[g.diff_source == 0]; used = set()
            for i_, r in gd.iterrows():
                cand = gs[~gs.index.isin(used)]
                if not len(cand): continue
                j_ = (cand.partner_miss - r.partner_miss).abs().idxmin()
                if abs(cand.partner_miss[j_] - r.partner_miss) <= 0.10 * max(1, (cand.partner_miss[j_] + r.partner_miss) / 2): idx_d.append(i_); idx_s.append(j_); used.add(j_)
        if idx_d:
            dm = (NET[:, idx_d].mean(1) - NET[:, idx_s].mean(1)) / Wt * 1e4
            res[lab]["strength_matched_diff_minus_same_per_10k"] = ci(dm); res[lab]["strength_matched_pairs"] = len(idx_d)
        else: res[lab]["strength_matched_diff_minus_same_per_10k"] = None; res[lab]["strength_matched_pairs"] = 0
        # adjusted regression with anchor fixed effects, on per-10k net, bootstrap over the same resamples
        Um = U[m].reset_index(drop=True); cols = np.flatnonzero(m)
        A_fe = pd.get_dummies(Um.anchor.astype(str)).to_numpy(float)
        X = np.column_stack([Um.diff_source.to_numpy(float), (Um.partner_miss - Um.partner_miss.mean()).to_numpy(float) / 1e2, A_fe])
        coef = [float(np.linalg.lstsq(X, NET[b, cols] / Wt[b] * 1e4, rcond=None)[0][0]) for b in range(B + 1)]
        res[lab]["adjusted_diff_source_coef_per_10k"] = ci(coef)
    R4[oname] = res
    print(f"\n=== {oname} ===")
    for lab, r in res.items():
        print(f"{lab:16s} diff {fmt(r['diff_source_net_per_10k'])} | same {fmt(r['same_source_net_per_10k'])} | diff-same {fmt(r['diff_minus_same_per_10k'])} | relative diff-same {fmt(r['diff_minus_same_relative'], 3)}"
              f" | matched {fmt(r['strength_matched_diff_minus_same_per_10k']) if r['strength_matched_diff_minus_same_per_10k'] else 'none'} (n={r['strength_matched_pairs']}) | adjusted {fmt(r['adjusted_diff_source_coef_per_10k'])}"
              f" | partner misses diff/same {r['partner_miss_mean_diff_vs_same'][0]:.0f}/{r['partner_miss_mean_diff_vs_same'][1]:.0f}")
REC["R4_fixed_anchor_source_brset"] = R4


=== referable ===
all_anchors      diff 35.54 (29.63 to 41.66) | same 18.64 (15.18 to 22.08) | diff-same 16.90 (12.57 to 21.42) | relative diff-same 0.085 (0.063 to 0.108) | matched none (n=0) | adjusted 16.90 (12.57 to 21.42) | partner misses diff/same 324/324
eyepacs_anchors  diff -25.47 (-31.32 to -20.00) | same 16.46 (12.30 to 20.53) | diff-same -41.93 (-48.43 to -35.07) | relative diff-same -0.303 (-0.359 to -0.253) | matched none (n=0) | adjusted 0.67 (-7.62 to 9.87) | partner misses diff/same 423/225
ddr_anchors      diff 96.56 (84.43 to 108.72) | same 20.83 (16.02 to 25.91) | diff-same 75.73 (64.20 to 87.32) | relative diff-same 0.291 (0.256 to 0.325) | matched none (n=0) | adjusted 11.20 (-2.35 to 24.71) | partner misses diff/same 225/423

=== severe ===
all_anchors      diff 24.62 (20.20 to 29.31) | same 10.64 (8.10 to 13.32) | diff-same 13.99 (10.29 to 17.71) | relative diff-same 0.199 (0.152 to 0.247) | matched none (n=0) | adjusted 13.99 (10.29 to 17.71) | partner misses 

## R7. Retinal degradation fingerprints, corrected model bootstrap
The original analysis drew models with replacement and then took the set of drawn names, discarding repeats. Here each resample draws models with replacement and **keeps multiplicities**: a pair of distinct models (a, b) enters with weight equal to the number of times a was drawn times the number of times b was drawn (rows are replicated accordingly); a model paired with a copy of itself contributes nothing, since self-pairs do not exist. A leave-one-model-out jackknife range is reported as a second view. The original set-based intervals are printed alongside.

In [8]:
from scipy.stats import spearmanr
PT = pd.read_csv(f"{FULL}/pair_table.csv"); Y = "comiss_sev"; D7 = PT.dropna(subset=[Y]).reset_index(drop=True)
NAMES7 = sorted(set(D7.a) | set(D7.b)); FPRED = [c for c in ["sim_frequency", "sim_channel", "sim_degradation", "sim_occlusion", "cka_trained", "cka_pretrained", "sim_pretrained_shift", "sim_clean_scores"] if c in D7.columns]
def partial_rho(df, p):
    Z_ = np.column_stack([np.ones(len(df)), rankdata(df.mean_sevmiss), rankdata(df.gap_sevmiss)])
    rx = rankdata(df[p]) - Z_ @ np.linalg.lstsq(Z_, rankdata(df[p]), rcond=None)[0]; ry = rankdata(df[Y]) - Z_ @ np.linalg.lstsq(Z_, rankdata(df[Y]), rcond=None)[0]
    return float(np.corrcoef(rx, ry)[0, 1])
rng7 = np.random.default_rng(20261030); ia = D7.a.map({n: i for i, n in enumerate(NAMES7)}).to_numpy(); ib = D7.b.map({n: i for i, n in enumerate(NAMES7)}).to_numpy()
draws = [np.bincount(rng7.integers(0, len(NAMES7), len(NAMES7)), minlength=len(NAMES7)) for _ in range(1000)]
old7 = json.load(open(f"{FULL}/ballistics_fullrecipe_results.json"))["primary"]["fingerprint_vs_severe_comiss"]
R7 = {}
for p in FPRED:
    obs = partial_rho(D7, p); bs = []
    for c in draws:
        w = c[ia] * c[ib]
        if (w > 0).sum() > 15: bs.append(partial_rho(D7.loc[np.repeat(np.arange(len(D7)), w)], p))
    jk = [partial_rho(D7[(D7.a != n) & (D7.b != n)], p) for n in NAMES7]
    R7[p] = {"partial_rho_given_strength": round(obs, 3), "model_bootstrap_ci95_with_multiplicity": [round(float(np.percentile(bs, 2.5)), 3), round(float(np.percentile(bs, 97.5)), 3)],
             "leave_one_model_out_range": [round(min(jk), 3), round(max(jk), 3)], "original_set_based_ci95": old7.get(p, {}).get("partial_ci95")}
    print(f"{p:22s} {R7[p]}")
REC["R7_ballistics_corrected_bootstrap"] = {"models": len(NAMES7), "pairs": int(len(D7)), "results": R7}

sim_frequency          {'partial_rho_given_strength': 0.303, 'model_bootstrap_ci95_with_multiplicity': [0.109, 0.495], 'leave_one_model_out_range': [0.258, 0.343], 'original_set_based_ci95': [0.109, 0.465]}
sim_channel            {'partial_rho_given_strength': 0.189, 'model_bootstrap_ci95_with_multiplicity': [-0.115, 0.46], 'leave_one_model_out_range': [0.103, 0.25], 'original_set_based_ci95': [-0.072, 0.422]}
sim_degradation        {'partial_rho_given_strength': 0.406, 'model_bootstrap_ci95_with_multiplicity': [0.178, 0.597], 'leave_one_model_out_range': [0.362, 0.464], 'original_set_based_ci95': [0.232, 0.567]}
sim_occlusion          {'partial_rho_given_strength': 0.016, 'model_bootstrap_ci95_with_multiplicity': [-0.2, 0.247], 'leave_one_model_out_range': [-0.052, 0.058], 'original_set_based_ci95': [-0.159, 0.211]}
cka_trained            {'partial_rho_given_strength': 0.089, 'model_bootstrap_ci95_with_multiplicity': [-0.269, 0.422], 'leave_one_model_out_range': [0.017, 0.207], 'origi

## Write the results record

In [9]:
REC["written_utc"] = datetime.datetime.now(datetime.timezone.utc).isoformat(); REC["input_sha256"] = HASHES
def tolist(x): return {str(k): tolist(v) for k, v in x.items()} if isinstance(x, dict) else ([tolist(v) for v in x] if isinstance(x, (list, tuple)) else (x.tolist() if isinstance(x, np.ndarray) else x))
with open(OUT_JSON, "w") as fh: json.dump(tolist(REC), fh, indent=1, default=float)
drive.flush_and_unmount(); drive.mount("/content/drive"); print("written", OUT_JSON)

Mounted at /content/drive
written /content/drive/MyDrive/stochastic_orthogonality/data/dr_ensemble_v2/04_results/revisions_c1/retina_revisions_c1_a.json
